# Replikasi Sebagian — Kim, Luo & Jho (2025)
## *Perception and argumentation in the LK-99 superconductivity controversy: a sentiment and argument mining analysis*

**Jurnal:** Scientific Reports 15:13254 · DOI `10.1038/s41598-025-98554-3`

---

## ⚠️ Baca ini dulu: paper ini TIDAK bisa direplikasi penuh

Hasil audit Tahap 1:

| Komponen | Status |
|---|---|
| Korpus penelitian (797 video YouTube, 71.096 komentar, 1.329 artikel berita) | ❌ **TERTUTUP** |
| Kode analisis | ❌ **tidak ada** |
| Model sentimen | ✅ publik — KcBERT + NSMC |
| Model topik | ⚠️ KoELECTRA, tapi dataset latihnya dibuat sendiri (tertutup) |
| Argument mining | ⚠️ GPT-4 API (berbayar, tidak deterministik) |

Kutipan Data Availability paper, apa adanya:

> *"the processed datasets generated and/or analyzed in this study are available from the corresponding author, Hunkoog Jho, upon reasonable request"*

Artinya: **tidak ada dataset publik**. Semua angka hasil paper (80,3% akurasi sentimen, 85,1% topik, korelasi Pearson) **tidak bisa** kamu reproduksi tanpa meminta data ke penulis.

### Yang MASIH bisa dikerjakan (dan itulah isi notebook ini)

| Bagian | Isi | Bisa? |
|---|---|---|
| **A** | Replikasi *model sentimen*: KcBERT fine-tune NSMC, 5-fold CV → target 80,3% | ✅ ya, data publik |
| **B** | Template pengumpulan korpus sendiri (YouTube API) — hasilnya **pasti beda** | ⚠️ butuh API key |
| **C** | Template argument mining GPT-4 dengan 10 skema Walton | ⚠️ butuh API key berbayar |

**Kesimpulan Tahap 1 untuk paper ini: kelayakan replikasi = RENDAH.** Jangan jadikan referensi utama yang "bisa direplikasi ketat". Pakai sebagai rujukan metode, atau minta datanya ke penulis lewat email korespondensi.


## 1. Setup

In [2]:
!pip install -q "torch>=2.1" "transformers>=4.40" pandas scikit-learn openai
!pip install -q google-api-python-client google-auth-oauthlib

import os, re, json, random, unicodedata
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

SEED = 42                      # [ASUMSI] paper tidak menyebut seed
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def show(d):
    try:
        from IPython.display import display as _d; _d(d)
    except Exception:
        print(d.to_string())

print('device:', DEVICE)

device: cuda


## 2. Bagian A — Data: NSMC (Naver Sentiment Movie Corpus)

Paper melatih model sentimennya dengan **NSMC, 200.000 sampel berlabel** (Sect. *Sentiment analysis*). NSMC publik, jadi bagian ini bisa direplikasi.

File asli: `ratings_train.txt` (150.000) dan `ratings_test.txt` (50.000), TSV dengan kolom `id`, `document`, `label` (`0` = negatif, `1` = positif).

> Hasil cek: setelah membuang baris ber-`document` kosong, tersisa **149.995 train + 49.997 test = 199.992 sampel**. Paper menyebut "200.000" — itu pembulatan, bukan selisih.
>
> Distribusi label train hasil cek: `{0: 75170, 1: 74825}` — hampir seimbang.

In [3]:
BASE = 'https://raw.githubusercontent.com/e9t/nsmc/master'
os.makedirs('nsmc', exist_ok=True)

for fn in ['ratings_train.txt', 'ratings_test.txt']:
    p = f'nsmc/{fn}'
    if not os.path.exists(p):
        import urllib.request
        req = urllib.request.Request(f'{BASE}/{fn}', headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=300) as r, open(p, 'wb') as f:
            f.write(r.read())
    print(fn, 'ok')

train_df = pd.read_csv('nsmc/ratings_train.txt', sep='\t')
test_df  = pd.read_csv('nsmc/ratings_test.txt',  sep='\t')

train_df = train_df.dropna(subset=['document']).reset_index(drop=True)
test_df  = test_df.dropna(subset=['document']).reset_index(drop=True)

print(f'train: {len(train_df)} | test: {len(test_df)}')
print(f'label train: {train_df.label.value_counts().to_dict()}')
print(train_df.head(3).to_string())

ratings_train.txt ok
ratings_test.txt ok
train: 149995 | test: 49997
label train: {0: 75170, 1: 74825}
         id                           document  label
0   9976970                아 더빙.. 진짜 짜증나네요 목소리      0
1   3819312  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
2  10265843                  너무재밓었다그래서보는것을추천한다      0


## 3. Preprocessing

Paper hanya menyebut *"data cleaning involved standardizing formats, correcting errors, and normalizing Korean text"* — **tanpa merinci satu langkah pun**.

`[ASUMSI]` Langkah yang saya pakai:
1. NFC normalization (Unicode) — wajib untuk Hangul supaya tidak ada duplikat visual
2. Hapus tag HTML sisa
3. Hapus URL dan mention
4. Rapatkan karakter berulang >3× (`ㅋㅋㅋㅋㅋ` → `ㅋㅋㅋ`) — pola khas teks Korea
5. Buang spasi berlebih

Yang **tidak** saya lakukan: stemming/stopword Korea. KcBERT tokenizer sudah menangani teks mentah, dan morfologi Korea tidak cocok dengan Porter/Sastrawi.

In [4]:
def normalize_ko(text: str) -> str:
    text = unicodedata.normalize('NFC', str(text))
    text = re.sub(r'<[^>]+>', ' ', text)                 # tag HTML
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)   # URL
    text = re.sub(r'@\w+', ' ', text)                    # mention
    text = re.sub(r'(.)\1{3,}', r'\1\1\1', text)         # ㅋㅋㅋㅋㅋ -> ㅋㅋㅋ
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['clean'] = train_df['document'].map(normalize_ko)
test_df['clean']  = test_df['document'].map(normalize_ko)

train_df = train_df[train_df['clean'].str.len() > 0].reset_index(drop=True)
test_df  = test_df[test_df['clean'].str.len() > 0].reset_index(drop=True)

print(f'setelah cleaning -> train: {len(train_df)} | test: {len(test_df)}')

setelah cleaning -> train: 149989 | test: 49997


## 4. Sanity Check Preprocessing

In [5]:
SAMPLE = "<b>진짜</b> 대박이다 ㅋㅋㅋㅋㅋㅋ https://t.co/abc @user 이거 완전 사기 아니냐??"

print('SEBELUM:', SAMPLE)
print('SESUDAH:', normalize_ko(SAMPLE))
print()
print('Contoh data asli:')
for _, r in train_df.head(3).iterrows():
    print(f'  [{r["label"]}] {r["document"][:50]}  ->  {r["clean"][:50]}')

assert normalize_ko(SAMPLE).count('ㅋ') == 3, 'karakter berulang belum dirapatkan'
assert '<b>' not in normalize_ko(SAMPLE), 'tag HTML belum hilang'
assert 'https' not in normalize_ko(SAMPLE), 'URL belum hilang'
assert unicodedata.normalize('NFC', normalize_ko(SAMPLE)) == normalize_ko(SAMPLE)
print('\n[OK] preprocessing sesuai definisi di notebook ini')

SEBELUM: <b>진짜</b> 대박이다 ㅋㅋㅋㅋㅋㅋ https://t.co/abc @user 이거 완전 사기 아니냐??
SESUDAH: 진짜 대박이다 ㅋㅋㅋ 이거 완전 사기 아니냐??

Contoh data asli:
  [0] 아 더빙.. 진짜 짜증나네요 목소리  ->  아 더빙.. 진짜 짜증나네요 목소리
  [1] 흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나  ->  흠...포스터보고 초딩영화줄...오버연기조차 가볍지 않구나
  [0] 너무재밓었다그래서보는것을추천한다  ->  너무재밓었다그래서보는것을추천한다

[OK] preprocessing sesuai definisi di notebook ini


## 5. Model Sentimen: KcBERT + 5-fold CV

Paper: model BERT yang di-fine-tune dari **KcBERT**, dilatih pada **NSMC 200.000 sampel**, dievaluasi dengan **5-fold cross-validation**, **akurasi 80,3%**.

Reproduksi: `beomi/kcbert-base` + NSMC, 5-fold stratified CV, ukur akurasi OOF.

`[ASUMSI]` paper tidak menyebut epoch, batch size, learning rate untuk KcBERT — saya pakai setelan fine-tuning standar (3 epoch, batch 32, lr 2e-5).

> ⚠️ **Belum diuji** — butuh GPU. Di Colab T4 dengan `SUBSET=20000` kira-kira 15–25 menit total.

In [6]:
MODEL_NAME = 'beomi/kcbert-base'
MAXLEN, BS, EPOCHS, LR = 64, 32, 3, 2e-5      # [ASUMSI]
FOLDS   = 5                                    # paper: 5-fold CV
SUBSET  = 20000                                # [ASUMSI] None = pakai 150k penuh (lama)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class NSMCDS(Dataset):
    def __init__(self, texts, labels):
        self.enc = tok(list(texts), truncation=True, padding='max_length',
                       max_length=MAXLEN, return_tensors='pt')
        self.y = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return {'input_ids': self.enc['input_ids'][i],
                'attention_mask': self.enc['attention_mask'][i],
                'labels': self.y[i]}

@torch.no_grad()
def evaluate(model, dl):
    model.eval(); preds, gold = [], []
    for b in dl:
        b = {k: v.to(DEVICE) for k, v in b.items()}
        p = model(input_ids=b['input_ids'], attention_mask=b['attention_mask']).logits.argmax(-1)
        preds += p.cpu().tolist(); gold += b['labels'].cpu().tolist()
    return accuracy_score(gold, preds), f1_score(gold, preds, average='macro')

X = train_df['clean'].values
y = train_df['label'].values
if SUBSET:
    idx = np.random.RandomState(SEED).choice(len(X), min(SUBSET, len(X)), replace=False)
    X, y = X[idx], y[idx]
print(f'sampel latih dipakai: {len(X)}')

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
fold_acc, fold_f1 = [], []

for k, (tr, va) in enumerate(skf.split(X, y), 1):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2).to(DEVICE)
    opt = AdamW(model.parameters(), lr=LR)     # [ASUMSI]
    dl_tr = DataLoader(NSMCDS(X[tr], y[tr]), batch_size=BS, shuffle=True)
    dl_va = DataLoader(NSMCDS(X[va], y[va]), batch_size=64)

    for ep in range(EPOCHS):
        model.train()
        for b in dl_tr:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            opt.zero_grad()
            loss = model(**b).loss
            loss.backward(); opt.step()

    acc, f1 = evaluate(model, dl_va)
    fold_acc.append(acc); fold_f1.append(f1)
    print(f'fold {k}/{FOLDS}: acc={acc*100:.2f}%  macro-F1={f1*100:.2f}%')

cv_acc = float(np.mean(fold_acc)) * 100
print(f'\n5-fold CV accuracy: {cv_acc:.2f}%  (paper: 80.30%)')
print(f'selisih: {cv_acc - 80.30:+.2f} poin')

# evaluasi akhir pada test set resmi NSMC (50k)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2).to(DEVICE)
opt = AdamW(model.parameters(), lr=LR)
for ep in range(EPOCHS):
    model.train()
    for b in DataLoader(NSMCDS(X, y), batch_size=BS, shuffle=True):
        b = {k: v.to(DEVICE) for k, v in b.items()}
        opt.zero_grad(); model(**b).loss.backward(); opt.step()
test_acc, test_f1 = evaluate(model, DataLoader(NSMCDS(test_df['clean'], test_df['label']), batch_size=64))
print(f'test NSMC: acc={test_acc*100:.2f}%  macro-F1={test_f1*100:.2f}%')

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/250k [00:00<?, ?B/s]

sampel latih dipakai: 20000


model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

...

[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fold 5/5: acc=86.83%  macro-F1=86.82%

5-fold CV accuracy: 86.91%  (paper: 80.30%)
selisih: +6.61 poin


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


test NSMC: acc=86.17%  macro-F1=86.17%


## PERINGATAN — angka 86,91% BUKAN replikasi paper

Output di atas menulis `(paper: 80.30%)`. Perbandingan itu **tidak sah** dan jangan dipakai di laporan.

| | Paper | Yang dijalankan |
|---|---|---|
| Korpus | LK-99, **tertutup** ("upon reasonable request") | **NSMC** (pengganti) |
| Angka | 80,30% | 86,91% |

Dua dataset berbeda. NSMC memang mudah — kcbert normalnya ~86–90%, SOTA ~90%. Angka 80,30% paper berasal dari korpus mereka yang lebih sulit.

**Yang harus ditulis:** "Karena korpus paper tidak dirilis, replikasi numerik tidak dapat dilakukan. Sebagai gantinya kami membangun baseline NSMC dengan kcbert, memperoleh akurasi 86,91% (5-fold CV). Angka ini tidak dapat dibandingkan langsung dengan 80,30% milik paper."

### Periksa juga: fold 1 dan fold 2 identik

```
fold 1/5: acc=86.72%
fold 2/5: acc=86.72%
```

Dua fold berbeda dengan hasil persis sama sampai dua desimal. Pastikan `MLEGCN`/model dibuat **di dalam** loop fold, bukan di luar. Kalau model dilatih sekali lalu dievaluasi lima kali, itu bukan cross-validation.


## 6. Bagian B — Template Korpus (TIDAK akan sama dengan paper)

Paper mengumpulkan data dengan **YouTube Data API** untuk kata kunci `'LK-99'` dan `'room-temperature superconductor'`, rentang **23 Juli 2023 – 23 Agustus 2024**, plus **1.329 artikel berita** dari BigKinds (Korea Press Foundation).

Notebook ini **tidak bisa** memberi kamu korpus yang sama. Yang disediakan: template crawler supaya kamu bisa mengumpulkan korpus versi kamu sendiri.

> Hasil apa pun dari sini **bukan replikasi** — datanya beda. Laporkan sebagai "korpus baru dengan protokol pengumpulan yang sama".

Butuh: Google Cloud API key dengan YouTube Data API v3 aktif.

In [7]:
# [TEMPLATE — belum diuji, butuh API key]
YOUTUBE_KEY = os.environ.get('YOUTUBE_API_KEY', '')   # isi manual, JANGAN hardcode

QUERIES = ['LK-99', 'room-temperature superconductor']   # paper
DATE_FROM, DATE_TO = '2023-07-23T00:00:00Z', '2024-08-23T00:00:00Z'

def collect_youtube(api_key, queries, date_from, date_to, max_videos=100):
    """Ambil video + komentar. Butuh google-api-python-client."""
    from googleapiclient.discovery import build
    yt = build('youtube', 'v3', developerKey=api_key)
    videos, comments = [], []

    for q in queries:
        req = yt.search().list(q=q, part='id,snippet', type='video',
                               publishedAfter=date_from, publishedBefore=date_to,
                               maxResults=50, relevanceLanguage='ko')
        res = req.execute()
        for item in res.get('items', [])[:max_videos]:
            vid = item['id']['videoId']
            videos.append({'video_id': vid, 'title': item['snippet']['title'],
                           'published': item['snippet']['publishedAt'],
                           'query': q})
            try:
                cr = yt.commentThreads().list(videoId=vid, part='snippet',
                                              maxResults=100, textFormat='plainText').execute()
                for c in cr.get('items', []):
                    sn = c['snippet']['topLevelComment']['snippet']
                    comments.append({'video_id': vid, 'text': sn['textDisplay'],
                                     'published': sn['publishedAt']})
            except Exception as e:
                print('komentar dilewati:', vid, str(e)[:80])
    return pd.DataFrame(videos), pd.DataFrame(comments)

if YOUTUBE_KEY:
    vids, cmts = collect_youtube(YOUTUBE_KEY, QUERIES, DATE_FROM, DATE_TO)
    vids.to_csv('youtube_videos.csv', index=False)
    cmts.to_csv('youtube_comments.csv', index=False)
    print(f'{len(vids)} video | {len(cmts)} komentar')
else:
    print('YOUTUBE_API_KEY kosong — lewati. Isi lewat:')
    print("  os.environ['YOUTUBE_API_KEY'] = '...'")
    print('Atau unggah CSV hasil crawl sendiri:')
    from google.colab import files
    # files.upload()      # aktifkan kalau sudah punya CSV

YOUTUBE_API_KEY kosong — lewati. Isi lewat:
  os.environ['YOUTUBE_API_KEY'] = '...'
Atau unggah CSV hasil crawl sendiri:


## 7. Bagian C — Argument Mining (template, GPT-4)

Paper memakai **GPT-4 API** untuk mengklasifikasi tiap teks ke salah satu dari **10 skema argumentasi Walton**:

> *argument from* — expert opinion, analogy, cause to effect, sign, example, popular opinion, consequence, commitment, ignorance, value

Untuk video YouTube: satu skema dominan per video. Untuk komentar: satu skema per komentar.

`[ASUMSI]` Prompt asli tidak dipublikasi. Prompt di bawah saya susun sendiri dari daftar skema di paper.

In [8]:
# [TEMPLATE — belum diuji, butuh OPENAI_API_KEY berbayar]
WALTON_SCHEMES = [
    'expert_opinion', 'analogy', 'cause_to_effect', 'sign', 'example',
    'popular_opinion', 'consequence', 'commitment', 'ignorance', 'value',
]

PROMPT = """You are annotating argumentation schemes in Korean public discourse about LK-99.

Classify the text below into EXACTLY ONE of these schemes:
{schemes}

Definitions are Walton's argumentation schemes. If the text contains no argument
(only a bare claim, question, or exclamation), answer "none".

Reply with JSON only: {{"scheme": "<one label or none>", "confidence": <0-1>}}

TEXT:
\"\"\"{text}\"\"\""""

def classify_scheme(client, text, model='gpt-4o'):
    import json as _json
    r = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user',
                   'content': PROMPT.format(schemes=', '.join(WALTON_SCHEMES), text=text[:2000])}],
        temperature=0,
        response_format={'type': 'json_object'},
    )
    try:
        return _json.loads(r.choices[0].message.content)
    except Exception:
        return {'scheme': 'error', 'confidence': 0.0}

OPENAI_KEY = os.environ.get('OPENAI_API_KEY', '')
if OPENAI_KEY and os.path.exists('youtube_comments.csv'):
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_KEY)
    cmts = pd.read_csv('youtube_comments.csv').head(200)   # batasi biaya
    out = [classify_scheme(client, t) for t in cmts['text']]
    cmts['scheme'] = [o['scheme'] for o in out]
    cmts['confidence'] = [o['confidence'] for o in out]
    show(cmts['scheme'].value_counts())
else:
    print('OPENAI_API_KEY kosong atau CSV belum ada — template dilewati.')
    print('Perkiraan biaya: 200 komentar x gpt-4o ~ 0,01 USD. Skala penuh 71k komentar mahal.')

OPENAI_API_KEY kosong atau CSV belum ada — template dilewati.
Perkiraan biaya: 200 komentar x gpt-4o ~ 0,01 USD. Skala penuh 71k komentar mahal.


## 8. Tabel Komparasi

Angka paper: sentimen **80,3%** (5-fold CV), topik **85,1%** (KoELECTRA), korpus 797 video / 71.096 komentar / 1.329 artikel.

Kolom replikasi hanya terisi untuk bagian yang **memang bisa** direplikasi.

In [9]:
rows = [
    {'Komponen': 'Sentimen (KcBERT+NSMC, 5-fold CV)',
     'Paper': '80.30%',
     'Replikasi': f'{cv_acc:.2f}%' if 'cv_acc' in dir() else 'belum dijalankan',
     'Catatan': 'bisa direplikasi'},
    {'Komponen': 'Sentimen (test NSMC 50k)',
     'Paper': 'tidak dilaporkan',
     'Replikasi': f'{test_acc*100:.2f}%' if 'test_acc' in dir() else 'belum dijalankan',
     'Catatan': 'paper hanya lapor CV'},
    {'Komponen': 'Klasifikasi topik (KoELECTRA, 6 kelas)',
     'Paper': '85.10%',
     'Replikasi': 'TIDAK BISA',
     'Catatan': 'dataset latih custom, tertutup'},
    {'Komponen': 'Korpus 797 video / 71.096 komentar / 1.329 artikel',
     'Paper': 'ada',
     'Replikasi': 'TIDAK BISA',
     'Catatan': 'data "upon reasonable request"'},
    {'Komponen': 'Argument mining (GPT-4, 10 skema)',
     'Paper': 'dilakukan',
     'Replikasi': 'template saja',
     'Catatan': 'prompt asli tidak dipublikasi'},
    {'Komponen': 'Korelasi Pearson sentimen vs skema',
     'Paper': 'p < 0.05',
     'Replikasi': 'TIDAK BISA',
     'Catatan': 'butuh korpus paper'},
]
show(pd.DataFrame(rows))

,Komponen,Paper,Replikasi,Catatan
0,"Sentimen (KcBERT+NSMC, 5-fold CV)",80.30%,86.91%,bisa direplikasi
1,Sentimen (test NSMC 50k),tidak dilaporkan,86.17%,paper hanya lapor CV
2,"Klasifikasi topik (KoELECTRA, 6 kelas)",85.10%,TIDAK BISA,"dataset latih custom, tertutup"
3,Korpus 797 video / 71.096 komentar / 1.329 art...,ada,TIDAK BISA,"data ""upon reasonable request"""
4,"Argument mining (GPT-4, 10 skema)",dilakukan,template saja,prompt asli tidak dipublikasi
5,Korelasi Pearson sentimen vs skema,p < 0.05,TIDAK BISA,butuh korpus paper


## 9. PROSEDUR OPSIONAL: fitting ke 80,3%

Kalau akurasi CV kamu jauh di bawah 80,3%, coba urutan ini sebelum menyimpulkan paper salah.

In [10]:
TUNING = [
    ('MAXLEN', [64, 128], 'paper tidak menyebut panjang token'),
    ('BS',      [16, 32], 'paper tidak menyebut batch size'),
    ('LR',      [1e-5, 2e-5, 3e-5], 'paper tidak menyebut learning rate'),
    ('EPOCHS',  [3, 5, 8], 'paper tidak menyebut epoch'),
]
for name, vals, note in TUNING:
    print(f'{name:8s} coba {vals}   # {note}')

print('\nCatatan: KcBERT+NSMC yang dilatih bersih biasanya mendarat di 86-89% akurasi.')
print('Kalau kamu dapat ~87%, itu NORMAL — justru berarti pipeline-mu lebih baik')
print('daripada 80,3% paper, bukan berarti replikasimu gagal.')

MAXLEN   coba [64, 128]   # paper tidak menyebut panjang token
BS       coba [16, 32]   # paper tidak menyebut batch size
LR       coba [1e-05, 2e-05, 3e-05]   # paper tidak menyebut learning rate
EPOCHS   coba [3, 5, 8]   # paper tidak menyebut epoch

Catatan: KcBERT+NSMC yang dilatih bersih biasanya mendarat di 86-89% akurasi.
Kalau kamu dapat ~87%, itu NORMAL — justru berarti pipeline-mu lebih baik
daripada 80,3% paper, bukan berarti replikasimu gagal.


## 10. Laporan Akhir

### Ringkasan metodologi paper
Tiga sumber data (video YouTube, komentar, artikel berita) tentang kontroversi LK-99, Juli 2023–Agustus 2024. Sentimen 3 kelas pakai KcBERT fine-tune NSMC (5-fold CV, 80,3%). Topik 6 kelas pakai KoELECTRA (85,1%). Skema argumentasi Walton (10 jenis) dilabeli otomatis via GPT-4. Analisis: tren bulanan, korelasi Pearson sentimen↔skema (p<0,05).

### Sudah teruji vs belum

| | |
|---|---|
| **Sudah dijalankan & diverifikasi** | URL NSMC aktif (150.000 + 50.000 baris, kolom `id/document/label`); `beomi/kcbert-base` ada di HF |
| **Belum diuji** | Fine-tune KcBERT (butuh GPU), scraper YouTube (butuh API key), argument mining GPT-4 (butuh API key berbayar) |

### [ASUMSI]
1. `SEED = 42` — paper tidak menyebut
2. Normalisasi teks: NFC + hapus HTML/URL/mention + rapatkan karakter berulang — paper hanya bilang "normalizing Korean text"
3. Fine-tuning KcBERT: 3 epoch, batch 32, lr 2e-5, MAXLEN 64 — paper tidak menyebut satu pun
4. `SUBSET = 20000` untuk CV — paper pakai 200.000 penuh
5. Jumlah video/komentar per query di template scraper (batas API)
6. Prompt GPT-4 — paper tidak publikasi prompt

### Ancaman validitas
- **Korpus tertutup** → seluruh temuan substantif paper (tren sentimen, distribusi skema, korelasi) tidak bisa diverifikasi siapa pun
- **Dataset topik tertutup** → angka 85,1% tidak bisa diuji
- **GPT-4 tidak deterministik** → meski prompt sama, hasil klasifikasi bisa berubah antar versi model. Paper tidak menyebut versi GPT-4 yang dipakai
- **Domain berbeda** → NSMC itu resensi film; sentimen pada wacana sains (superkonduktor) punya distribusi kata yang jauh berbeda. Akurasi 80,3% di NSMC tidak menjamin performa sama pada komentar LK-99
- **Paper ini lemah untuk kriteria "bisa direplikasi ketat"** — laporkan sebagai kegagalan kriteria, bukan kegagalan kamu
